# Week 3 Day 4 — Daily Svensson Loop

Runs `fit_svensson_daily` over the full history and explores the resulting
time series of parameters.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from termstructure.curves.svensson import fit_svensson_daily, svensson_zero_rate

## 1. Run the full historical loop

This takes a few minutes for the full history (1961–present).
The result is saved to `data/processed/svensson_params.parquet`
so you only need to run it once.

In [ ]:
params_df = fit_svensson_daily('1961-06-14', '2024-12-31')
params_df.head()

## 2. Quick quality check

Median RMSE should be well under 1 bp (we're fitting to the Fed's already-smooth zero yields).
Any day above 5 bp is suspicious — probably a data gap or optimizer stumble.

In [ ]:
print(f'Dates fitted:      {len(params_df):,}')
print(f'RMSE median:       {params_df.rmse_bps.median():.3f} bp')
print(f'RMSE 95th pctile:  {params_df.rmse_bps.quantile(0.95):.3f} bp')
print(f'RMSE max:          {params_df.rmse_bps.max():.3f} bp')
print(f'Days > 5 bp:       {(params_df.rmse_bps > 5).sum()}')

## 3. Parameter time series

Each parameter has an economic meaning:
- **β0**: long-run rate — tracks secular decline from ~15% in 1982 to ~4% post-2022
- **β1**: slope loading — negative means upward-sloping curve (normal)
- **β2, β3**: hump shapes — harder to interpret alone, but together they shape the middle of the curve

Note: individual parameter values can be noisy because Svensson is ill-conditioned
(many parameter sets give equally good fits). The *curve* is stable even when
parameters vary — what matters for Week 5 is the fitted zero rate, not the params.

In [ ]:
params_df = params_df.set_index('date')

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(params_df.index, params_df.beta0 * 100, lw=0.7)
axes[0].set_ylabel('β0 (%)')
axes[0].set_title('β0 — long-run rate')
axes[0].grid(True, alpha=0.3)

axes[1].plot(params_df.index, params_df.beta1 * 100, lw=0.7, color='tomato')
axes[1].set_ylabel('β1 (%)')
axes[1].set_title('β1 — slope')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].grid(True, alpha=0.3)

axes[2].plot(params_df.index, params_df.rmse_bps, lw=0.5, color='gray')
axes[2].set_ylabel('RMSE (bp)')
axes[2].set_title('Fit quality — RMSE per day')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/week3_day4_params.png', dpi=150)
plt.show()

## 4. Compare our β0 to the Fed's published β0

The Fed's parameters are in the same parquet (columns beta0–beta3, tau1, tau2).
As noted in notebook 11, individual parameter values often differ because Svensson
is ill-conditioned. The comparison that *matters* is the fitted curve, not the params.
We'll do the full curve comparison in Day 5.

In [ ]:
fed = pd.read_parquet('../data/processed/treasury_bonds.parquet')
fed['date'] = pd.to_datetime(fed['date'])
fed = fed.set_index('date').sort_index()

common = params_df.index.intersection(fed.index)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(common, params_df.loc[common, 'beta0'] * 100, label='Our β0', lw=0.7)
ax.plot(common, fed.loc[common, 'beta0'], label="Fed's β0", lw=0.7, linestyle='--', color='tomato')
ax.set_title('β0 comparison: ours vs. Fed — parameter values differ but curve quality is the same')
ax.set_ylabel('β0 (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('β0 correlation:', params_df.loc[common, 'beta0'].corr(fed.loc[common, 'beta0']))